In [ ]:
# ============================================================
# 0. ENVIRONMENT
# ============================================================
import os, sys, subprocess, pathlib, socket, datetime, random, json, csv, math, warnings

IS_KAGGLE = os.path.exists("/kaggle/input")
WORK_DIR  = pathlib.Path("/kaggle/working") if IS_KAGGLE else pathlib.Path(".")
WORK_DIR.mkdir(parents=True, exist_ok=True)

import torch
import numpy as np

WORLD_SIZE = torch.cuda.device_count()
DEVICE     = "cuda" if WORLD_SIZE > 0 else "cpu"

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"GPUs     : {WORLD_SIZE}")
for i in range(WORLD_SIZE):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i} : {p.name}  {p.total_memory/1e9:.1f} GB")

print(f"Strategy : {'DDP mp.spawn' if WORLD_SIZE > 1 else 'single GPU' if WORLD_SIZE == 1 else 'CPU'}")

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


In [ ]:
# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================
for pkg in [
    "timm>=0.9.0",
    "opencv-python-headless>=4.8.0",
    "onnx>=1.14.0",
    "onnxruntime>=1.15.0",
    "grad-cam>=1.4.0",
    "pydicom>=2.4.0",
    "tqdm>=4.65.0",
    "wandb>=0.15.0",
]:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       capture_output=True, text=True)
    print(f"{'OK' if r.returncode == 0 else 'FAIL'} {pkg}")


In [ ]:
# ============================================================
# 2. CLONE SOURCE FROM GITHUB
# ============================================================
REPO      = "https://github.com/tripathiji1312/cough-vision.git"
CLONE_DIR = WORK_DIR / "cough-vision"

if not CLONE_DIR.exists():
    r = subprocess.run(["git", "clone", "--depth", "1", REPO, str(CLONE_DIR)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{r.stderr[-400:]}")
    print("Cloned OK")
else:
    print(f"Already at {CLONE_DIR}")

SRC_ROOT = CLONE_DIR / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from config import get_config
print(f"Config OK -- {get_config().cnn.name}")


In [ ]:
# ============================================================
# 3. DATASET PATHS  (auto-discovery with recursive search)
# ============================================================
import pathlib

KI = pathlib.Path("/kaggle/input") if IS_KAGGLE else pathlib.Path("data")

def find_dataset(*slugs):
    user = slugs[0] if len(slugs) >= 2 else None
    name = slugs[-1]
    candidates = [KI / name]
    if user:
        candidates += [KI / "datasets" / user / name,
                       KI / user / name,
                       KI / f"{user}-{name}"]
    for p in candidates:
        if p.exists():
            return p
    return None

def find_dir_recursive(root, *glob_patterns):
    """Return first directory inside root matching any of glob_patterns."""
    if root is None:
        return None
    for pat in glob_patterns:
        hits = sorted(root.rglob(pat))
        if hits:
            return hits[0].parent if hits[0].is_file() else hits[0]
    return None

def peek_textfile(path, n=3):
    """Return first n non-empty lines of a text file."""
    if path is None or not path.exists():
        return []
    try:
        lines = path.read_text(errors="replace").splitlines()
        return [l for l in lines if l.strip()][:n]
    except Exception:
        return []


# ── Dataset roots ─────────────────────────────────────────────────────────────
KMADER_ROOT = find_dataset("kmader",       "pulmonary-chest-xray-abnormalities")
TBX11K_ROOT = find_dataset("usmanshams",   "tbx-11")
TAWSIF_ROOT = find_dataset("tawsifurrahman","tuberculosis-tb-chest-xray-dataset")

# ── Kmader: search recursively for CHNCXR (Shenzhen) and MCUCXR (Montgomery) --
# The dataset may use ChinaSet_AllFiles/ or shenzhen/ depending on version
SHENZHEN_IMG   = find_dir_recursive(KMADER_ROOT, "CHNCXR_*.png", "CHNCXR_*_0.png")
MONTGOMERY_IMG = find_dir_recursive(KMADER_ROOT, "MCUCXR_*.png", "MCUCXR_*_0.png")

# ── TBX11K: list file + images directory ─────────────────────────────────────
TBX11K_LIST  = find_dir_recursive(TBX11K_ROOT, "TBX11K_trainval.txt")
if TBX11K_LIST and TBX11K_LIST.is_dir():
    TBX11K_LIST = TBX11K_LIST / "TBX11K_trainval.txt"
elif TBX11K_ROOT:
    # Search directly
    hits = sorted(TBX11K_ROOT.rglob("TBX11K_trainval.txt"))
    TBX11K_LIST = hits[0] if hits else None

TBX11K_IMGS_ROOT = None
if TBX11K_ROOT:
    hits = sorted(TBX11K_ROOT.rglob("imgs"))
    TBX11K_IMGS_ROOT = hits[0] if hits else None

TBX11K_ANN_ROOT = None
if TBX11K_ROOT:
    for jname in ["trainval.json", "train.json"]:
        hits = sorted(TBX11K_ROOT.rglob(jname))
        if hits:
            TBX11K_ANN_ROOT = hits[0]
            break

# ── Tawsif Normal ─────────────────────────────────────────────────────────────
TAWSIF_NORMAL = find_dir_recursive(TAWSIF_ROOT, "Normal-*.png")

DATA_DIR = WORK_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

print("Dataset roots:")
for name, p in [("kmader", KMADER_ROOT), ("tbx-11", TBX11K_ROOT), ("tawsif", TAWSIF_ROOT)]:
    print(f"  {'FOUND' if p else 'MISS '} {name}: {p}")

print("\nResolved subpaths:")
for name, p in [("Shenzhen imgs",    SHENZHEN_IMG),
                ("Montgomery imgs",  MONTGOMERY_IMG),
                ("TBX11K list",      TBX11K_LIST),
                ("TBX11K imgs/",     TBX11K_IMGS_ROOT),
                ("TBX11K JSON",      TBX11K_ANN_ROOT),
                ("Tawsif Normal/",   TAWSIF_NORMAL)]:
    print(f"  {'FOUND' if p and pathlib.Path(p).exists() else 'MISS '} {name}: {p}")

# Peek at list file format
if TBX11K_LIST and pathlib.Path(TBX11K_LIST).exists():
    sample = peek_textfile(TBX11K_LIST, 5)
    print(f"\nTBX11K list file first 5 lines: {sample}")

# Show extra kmader tree if Shenzhen still missing
if SHENZHEN_IMG is None and KMADER_ROOT:
    print("\nkmader tree (up to depth 4):")
    for p in sorted(KMADER_ROOT.rglob("*")):
        depth = len(p.relative_to(KMADER_ROOT).parts)
        if depth <= 4:
            print("  " * depth + p.name + ("/" if p.is_dir() else ""))


In [ ]:
# ============================================================
# 4. BUILD METADATA CSV
# ============================================================
import json as _json

records = []

# ------------------------------------------------------------------
# Dataset 1a: Shenzhen  (CHNCXR_XXXX_1.png = TB, _0 = Normal)
# ------------------------------------------------------------------
if SHENZHEN_IMG is not None and pathlib.Path(SHENZHEN_IMG).exists():
    for img in sorted(pathlib.Path(SHENZHEN_IMG).glob("*.png")):
        try:
            tb = int(img.stem.split("_")[-1])
        except ValueError:
            continue
        records.append({"image_path": str(img), "tb_label": tb,
                         "findings_label": "0,0,0,0,0,0", "active_inactive_label": -1,
                         "site": "shenzhen", "view_position": "PA"})
    print(f"Shenzhen  : {sum(r['site']=='shenzhen' for r in records)}")
else:
    print("Shenzhen  : MISSING")

# ------------------------------------------------------------------
# Dataset 1b: Montgomery (MCUCXR_XXXX_1.png = TB, _0 = Normal)
# ------------------------------------------------------------------
if MONTGOMERY_IMG is not None and pathlib.Path(MONTGOMERY_IMG).exists():
    n0 = len(records)
    for img in sorted(pathlib.Path(MONTGOMERY_IMG).glob("*.png")):
        try:
            tb = int(img.stem.split("_")[-1])
        except ValueError:
            continue
        records.append({"image_path": str(img), "tb_label": tb,
                         "findings_label": "0,0,0,0,0,0", "active_inactive_label": -1,
                         "site": "montgomery", "view_position": "PA"})
    print(f"Montgomery: {len(records)-n0}")
else:
    print("Montgomery: MISSING")

# ------------------------------------------------------------------
# Dataset 2: TBX11K
# ------------------------------------------------------------------
if TBX11K_ROOT is not None:

    # Step 1: determine list-file entry format + collect image paths
    tbx_files = []

    if TBX11K_LIST is not None and pathlib.Path(TBX11K_LIST).exists():
        lines = pathlib.Path(TBX11K_LIST).read_text().strip().splitlines()
        sample_line = lines[0].strip() if lines else ""
        print(f"TBX11K list: {len(lines)} entries, sample='{sample_line}'")

        # Try to resolve each line to a real file. Accept multiple candidate roots.
        roots_to_try = []
        if TBX11K_IMGS_ROOT:
            roots_to_try.append(pathlib.Path(TBX11K_IMGS_ROOT))
        if TBX11K_ROOT:
            tbx11k_sub = pathlib.Path(TBX11K_ROOT) / "TBX11K"
            if tbx11k_sub.exists():
                roots_to_try.append(tbx11k_sub)
            roots_to_try.append(pathlib.Path(TBX11K_ROOT))

        for line in lines:
            fname = line.strip()
            if not fname:
                continue
            found = False
            for root in roots_to_try:
                candidate = root / fname
                if candidate.exists():
                    tbx_files.append(candidate)
                    found = True
                    break
            # Also try just the basename in each split directory
            if not found and TBX11K_IMGS_ROOT:
                bname = pathlib.Path(fname).name
                for split in ["train", "val"]:
                    candidate = pathlib.Path(TBX11K_IMGS_ROOT) / split / bname
                    if candidate.exists():
                        tbx_files.append(candidate)
                        found = True
                        break

        print(f"TBX11K list -> {len(tbx_files)} images resolved on disk")

    # Fallback: scan train/ + val/ directly if list gave 0 results
    if len(tbx_files) == 0 and TBX11K_IMGS_ROOT is not None:
        imgs_root = pathlib.Path(TBX11K_IMGS_ROOT)
        for split in ["train", "val"]:
            split_dir = imgs_root / split
            if split_dir.exists():
                tbx_files.extend(sorted(split_dir.glob("*.png")))
                tbx_files.extend(sorted(split_dir.glob("*.jpg")))
        print(f"TBX11K: scanned train+val -> {len(tbx_files)} images")

    if tbx_files:
        # Step 2: load COCO JSON for bbox category annotations
        img_cats: dict = {}
        if TBX11K_ANN_ROOT is not None:
            try:
                data    = _json.loads(pathlib.Path(TBX11K_ANN_ROOT).read_text())
                id2fn   = {img["id"]: pathlib.Path(img["file_name"]).name
                           for img in data.get("images", [])}
                catmap  = {c["id"]: c["name"] for c in data.get("categories", [])}
                for ann in data.get("annotations", []):
                    fn  = id2fn.get(ann.get("image_id", -1), "")
                    cat = catmap.get(ann.get("category_id", -1), "")
                    if fn and cat:
                        img_cats.setdefault(fn, set()).add(cat)
                print(f"TBX11K JSON: {len(img_cats)} annotated images")
            except Exception as e:
                print(f"TBX11K JSON load failed: {e}")

        # Step 3: build records
        # Label strategy (no COCO JSON required):
        #   The path parent directory encodes the class:
        #     tb/   or active_tb/   or latent_tb/  -> TB positive
        #     health/ or sick/ or sick_non-tb/     -> Normal / negative
        #   If a COCO JSON WAS loaded, its annotations take priority.
        n0 = len(records)
        for img_p in tbx_files:
            # --- COCO JSON annotation (priority if available) ---
            cats = img_cats.get(img_p.name, set())
            TB_CATS = {c for c in cats if c.lower() not in
                       ("nofinding", "no_finding", "normal", "healthy")}

            if TB_CATS:
                tb_label = 1
                ai_label = (1 if "ActiveTuberculosis" in TB_CATS
                             else 0 if "ObsoletePulmonaryTuberculosis" in TB_CATS
                             else -1)
            else:
                # --- Infer label from parent directory name ---
                # sample path: .../TBX11K/imgs/tb/tb0003.png
                #              .../TBX11K/imgs/health/health0001.png
                parent = img_p.parent.name.lower()
                if parent in ("tb", "active_tb", "latent_tb",
                              "active&latent_tb", "uncertain_tb"):
                    tb_label = 1
                    ai_label = (0 if parent == "latent_tb" else
                                1 if "active" in parent else -1)
                elif parent.startswith("tb"):
                    tb_label = 1
                    ai_label = -1
                else:
                    # health/, sick/, sick_non-tb/, healthy/, normal/ -> 0
                    tb_label = 0
                    ai_label = -1
            records.append({"image_path": str(img_p), "tb_label": tb_label,
                             "findings_label": "0,0,0,0,0,0",
                             "active_inactive_label": ai_label,
                             "site": "tbx11k", "view_position": "PA"})
        n_added = len(records) - n0
        tb_pos  = sum(r["tb_label"]==1 for r in records[n0:])
        act     = sum(r["active_inactive_label"]==1 for r in records[n0:])
        print(f"TBX11K    : {n_added} imgs  TB+={tb_pos}  Active={act}")
    else:
        print("TBX11K    : no images found")
else:
    print("TBX11K    : MISSING")

# ------------------------------------------------------------------
# Dataset 3: Tawsif Normal only
# ------------------------------------------------------------------
if TAWSIF_NORMAL is not None and pathlib.Path(TAWSIF_NORMAL).exists():
    n0 = len(records)
    for img in pathlib.Path(TAWSIF_NORMAL).glob("*.png"):
        records.append({"image_path": str(img), "tb_label": 0,
                         "findings_label": "0,0,0,0,0,0", "active_inactive_label": -1,
                         "site": "tawsif_normal", "view_position": "PA"})
    print(f"Tawsif    : {len(records)-n0} Normal imgs")
else:
    print("Tawsif    : MISSING")

# Demo fallback
if not records:
    import cv2
    print("\nDEMO MODE")
    demo = DATA_DIR / "demo"; demo.mkdir(exist_ok=True)
    for i in range(200):
        arr = np.random.randint(20, 220, (256, 256), dtype=np.uint8)
        p = demo / f"img_{i:04d}.png"
        cv2.imwrite(str(p), arr)
        records.append({"image_path": str(p), "tb_label": int(i % 5 == 0),
                         "findings_label": "0,0,0,0,0,0", "active_inactive_label": -1,
                         "site": "demo", "view_position": "PA"})

print(f"\nTotal  : {len(records)}")
print(f"  TB+  : {sum(r['tb_label']==1 for r in records)}")
print(f"  Norm : {sum(r['tb_label']==0 for r in records)}")
print(f"  TB%  : {sum(r['tb_label']==1 for r in records)/max(1,len(records))*100:.1f}%")


In [ ]:
# ============================================================
# 5. STRATIFIED SPLIT + WRITE CSVs
# ============================================================
from data.dataset import stratified_split

train_recs, val_recs, test_recs = stratified_split(records, val_fraction=0.10, test_fraction=0.10, seed=42)

FIELDS = ["image_path","tb_label","findings_label","active_inactive_label","site","view_position","split"]

def write_csv(path, rows, spl):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        for r in rows:
            r["split"] = spl
            w.writerow({k: r.get(k, "") for k in FIELDS})

write_csv(DATA_DIR / "train.csv", train_recs, "train")
write_csv(DATA_DIR / "val.csv",   val_recs,   "val")
write_csv(DATA_DIR / "test.csv",  test_recs,  "test")

for name, recs in [("train", train_recs), ("val", val_recs), ("test", test_recs)]:
    print(f"{name:5}: {len(recs):>5}  TB+={sum(r['tb_label']==1 for r in recs)}  Normal={sum(r['tb_label']==0 for r in recs)}")


In [ ]:
# ============================================================
# 6. PREPROCESSING SANITY CHECK
# ============================================================
import matplotlib.pyplot as plt
import cv2
from data.preprocessing import load_image, apply_clahe, apply_gaussian_denoise, preprocess_cxr, passes_qc

s   = train_recs[0]
raw = load_image(s["image_path"])
clh = apply_clahe(raw, clip_limit=2.5)
dns = apply_gaussian_denoise(clh, sigma=0.5)
out = preprocess_cxr(raw, target_size=224)

print(f"Image : {s['image_path']}")
print(f"Label : {'TB' if s['tb_label'] else 'Normal'}")
print(f"Raw   : {raw.shape}  QC={'PASS' if passes_qc(raw) else 'FAIL'}")
print(f"Output: {out.shape}  {out.dtype}  min={out.min():.3f} max={out.max():.3f}")

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, img, title in zip(ax, [raw, clh, dns, out[0]], ["Raw", "CLAHE", "Denoised", "Net input"]):
    a.imshow(img, cmap="gray"); a.set_title(title); a.axis("off")
plt.tight_layout()
plt.savefig(WORK_DIR / "preprocessing_check.png", dpi=100)
plt.show()


In [ ]:
# ============================================================
# 7. HYPERPARAMETERS  (edit here)
# batch_size: single GPU -> effective = batch_size x accum_steps = 64
# ============================================================
HP = dict(
    preset          = "densenet_vit_ensemble",
    cnn_backbone    = "densenet121",
    vit_backbone    = "vit_small_patch16_384",
    pretrained      = "imagenet",
    max_epochs      = 60,
    freeze_epochs   = 3,
    batch_size      = 32,              # single GPU: 32x1x2 = 64 effective
    accum_steps     = 2,
    backbone_lr     = 1e-5,
    head_lr         = 1e-3,
    weight_decay    = 1e-4,
    warmup_epochs   = 5,
    patience        = 10,
    focal_gamma     = 2.0,
    focal_alpha     = 0.75,
    label_smoothing = 0.1,
    cls_weight      = 1.0,
    findings_weight = 0.3,
    active_weight   = 0.2,
    use_cutmix      = True,
    use_mixup       = True,
    pos_weight      = 5.0,
    who_sensitivity = 0.90,
    mixed_precision = True,
    seed            = 42,
    num_workers     = 2,
    port            = 12355,
)

eff = HP["batch_size"] * HP["accum_steps"]
print(f"Effective batch = {eff}  ({HP['batch_size']} x {HP['accum_steps']} accum steps)")
(WORK_DIR / "hyperparameters.json").write_text(json.dumps(HP, indent=2))


In [ ]:
# ============================================================
# 8. DDP UTILITIES
# ============================================================
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Sampler
from torch.utils.data.distributed import DistributedSampler


def find_free_port(preferred=12355):
    for port in [preferred] + list(range(49152, 49252)):
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
                s.bind(("", port))
                return port
        except OSError:
            continue
    raise RuntimeError("No free port found")


def setup_ddp(rank, world_size, port):
    os.environ.update({
        "MASTER_ADDR": "localhost",
        "MASTER_PORT": str(port),
        "NCCL_P2P_DISABLE": "1",   # Kaggle T4 uses PCIe, not NVLink
        "NCCL_IB_DISABLE":  "1",
    })
    dist.init_process_group(
        backend="nccl", rank=rank, world_size=world_size,
        timeout=datetime.timedelta(minutes=60),
    )
    torch.cuda.set_device(rank)


def cleanup_ddp():
    if dist.is_available() and dist.is_initialized():
        dist.destroy_process_group()


class DistributedWeightedSampler(Sampler):
    """Weighted sampling + DDP sharding: each rank gets a unique balanced shard."""
    def __init__(self, weights, num_replicas, rank, replacement=True, seed=42):
        self.weights      = torch.as_tensor(weights, dtype=torch.float64)
        self.num_replicas = num_replicas
        self.rank         = rank
        self.replacement  = replacement
        self.seed         = seed
        self.epoch        = 0
        self.num_samples  = math.ceil(len(weights) / num_replicas)
        self.total_size   = self.num_samples * num_replicas

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __iter__(self):
        g = torch.Generator()
        g.manual_seed(self.seed + self.epoch * 10007)
        idx = torch.multinomial(self.weights, self.total_size,
                                replacement=self.replacement, generator=g).tolist()
        return iter(idx[self.rank : self.total_size : self.num_replicas])

    def __len__(self):
        return self.num_samples


print("DDP utilities ready")


In [ ]:
# ============================================================
# 9. MODEL SMOKE-TEST  (single process, no DDP)
# ============================================================
from config import get_config
from models import build_ensemble

cfg = get_config(HP["preset"])
cfg.cnn.name         = HP["cnn_backbone"]
cfg.vit.name         = HP["vit_backbone"]
cfg.cnn.pretrained   = HP["pretrained"]
cfg.vit.pretrained   = HP["pretrained"]
cfg.train.output_dir = str(WORK_DIR / "checkpoints")

# Smoke test always runs on CPU (reviewer fix: avoid CUDA-before-fork
# which corrupts NCCL context when Cell 11 forks the DDP workers).
_dev = torch.device("cpu")
_m   = build_ensemble(cfg).to(_dev)

print(f"Params : {sum(p.numel() for p in _m.parameters())/1e6:.1f} M")

_m.eval()
with torch.no_grad():
    _o = _m(torch.zeros(2, 3, 224, 224, device=_dev))

print(f"Keys   : {list(_o.keys())}")
print(f"tb_prob: {_o['tb_prob'].tolist()}")
print("Forward pass OK")

del _m, _o
if _dev.type == "cuda":
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# 10. DDP TRAINING WORKER
#     One copy per GPU -- launched by mp.spawn or called directly
# ============================================================
def ddp_worker(rank, world_size, args):
    import os, sys, math, time, json, datetime, warnings, pathlib
    sys.path.insert(0, args["src_root"])

    setup_ddp(rank, world_size, args["port"])
    is_main = (rank == 0)
    device  = torch.device(f"cuda:{rank}")
    torch.manual_seed(args["seed"] + rank * 137)

    if is_main:
        eff = args["batch_size"] * world_size * args["accum_steps"]
        print(f"[DDP] {world_size} processes | batch/GPU={args['batch_size']} | effective={eff}")

    from config import get_config
    from models import build_ensemble
    from training.losses import MultiTaskLoss
    from training.scheduler import build_optimizer, cosine_schedule_with_warmup
    from data.dataset import TBDataset, compute_sample_weights
    from data.augmentation import get_train_transform, get_inference_transform, CutMixMixUpCollator
    from torch.utils.data import DataLoader
    from torch.utils.data.distributed import DistributedSampler
    import numpy as np

    _roc = None
    try:
        from sklearn.metrics import roc_auc_score as _roc
    except ImportError:
        pass

    cfg = get_config(args["preset"])
    cfg.cnn.name       = args["cnn_backbone"]
    cfg.vit.name       = args["vit_backbone"]
    cfg.cnn.pretrained = args["pretrained"]
    cfg.vit.pretrained = args["pretrained"]
    out_dir = pathlib.Path(args["output_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)

    model = build_ensemble(cfg).to(device)
    if world_size > 1:
        model = DDP(model, device_ids=[rank], output_device=rank, find_unused_parameters=False)

    root = pathlib.Path("/")
    train_ds = TBDataset(
        csv_path=args["train_csv"], image_root=root, split="train",
        transform=get_train_transform(image_size=224, rotation_degrees=10.0,
                                      translate_fraction=0.05, scale_range=(0.85, 1.15),
                                      brightness_jitter=0.20, contrast_jitter=0.20),
    )
    val_ds = TBDataset(
        csv_path=args["val_csv"], image_root=root, split="val",
        transform=get_inference_transform(image_size=224),
    )

    weights = compute_sample_weights([int(r.get("tb_label", 0)) for r in train_ds.records],
                                     pos_weight=args["pos_weight"])
    train_sampler = DistributedWeightedSampler(weights, world_size, rank, seed=args["seed"])
    val_sampler   = DistributedSampler(val_ds, num_replicas=world_size, rank=rank, shuffle=False)

    collator = CutMixMixUpCollator(
        n_classes=2, cutmix_alpha=1.0, mixup_alpha=0.4,
        cutmix_prob=0.5 if args["use_cutmix"] else 0.0,
        mixup_prob=0.5  if args["use_mixup"]  else 0.0,
    )
    nw = args["num_workers"]
    train_loader = DataLoader(train_ds, sampler=train_sampler, batch_size=args["batch_size"],
                              num_workers=nw, pin_memory=True, drop_last=True,
                              collate_fn=collator, persistent_workers=(nw > 0))
    val_loader   = DataLoader(val_ds, sampler=val_sampler, batch_size=args["batch_size"] * 2,
                              num_workers=nw, pin_memory=True, persistent_workers=(nw > 0))

    criterion = MultiTaskLoss(
        cls_weight=args["cls_weight"], findings_weight=args["findings_weight"],
        active_weight=args["active_weight"], focal_gamma=args["focal_gamma"],
        focal_alpha=args["focal_alpha"], label_smoothing=args["label_smoothing"],
    ).to(device)

    accum, max_ep, frz_ep = args["accum_steps"], args["max_epochs"], args["freeze_epochs"]
    wrm_ep, pat, mp_flag  = args["warmup_epochs"], args["patience"], args["mixed_precision"]

    model.freeze_backbones()
    optimizer    = build_optimizer(model, args["backbone_lr"], args["head_lr"], args["weight_decay"])
    steps_ep     = len(train_loader)
    scheduler    = cosine_schedule_with_warmup(optimizer, wrm_ep * steps_ep, max_ep * steps_ep)
    scaler       = torch.amp.GradScaler("cuda", enabled=mp_flag)

    best_auc, no_improve, history = 0.0, 0, []

    for epoch in range(max_ep):
        t0 = time.time()
        train_sampler.set_epoch(epoch)

        if epoch == frz_ep:
            model.unfreeze_backbones()
            optimizer = build_optimizer(model, args["backbone_lr"], args["head_lr"], args["weight_decay"])
            scheduler = cosine_schedule_with_warmup(optimizer, 0, max_ep * steps_ep)
            if is_main:
                print(f"[Epoch {epoch:03d}] Phase 2b: backbones unfrozen")

        # ---- train ----
        model.train()
        run_loss = torch.zeros(1, device=device)
        n_steps  = 0
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader):
            if isinstance(batch[1], dict):
                images    = batch[0].to(device, non_blocking=True)
                ld        = batch[1]
                tb_labels = ld["labels_a"].to(device, non_blocking=True)
                lam       = float(ld.get("lam", 1.0))
                labels_b  = ld["labels_b"].to(device, non_blocking=True)
                findings  = torch.zeros(images.shape[0], 6, device=device)
                active    = torch.full((images.shape[0],), -1, dtype=torch.long, device=device)
            else:
                images, tb_raw, find_raw, act_raw, *_ = batch
                images    = images.to(device, non_blocking=True)
                tb_labels = tb_raw.to(device, non_blocking=True)
                findings  = find_raw.to(device, non_blocking=True)
                active    = act_raw.to(device, non_blocking=True)
                lam, labels_b = 1.0, None

            with torch.amp.autocast("cuda", enabled=mp_flag):
                out  = model(images)
                loss = criterion(out, tb_labels, findings, active, lam=lam, labels_b=labels_b)["loss"] / accum

            scaler.scale(loss).backward()
            run_loss += loss.detach() * accum

            if (step + 1) % accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
                scheduler.step(); optimizer.zero_grad()
            n_steps += 1

        if len(train_loader) % accum != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            scheduler.step(); optimizer.zero_grad()

        dist.all_reduce(run_loss, op=dist.ReduceOp.AVG)
        avg_train = float(run_loss / max(1, n_steps))

        # ---- validate ----
        model.eval()
        vl_t = torch.zeros(1, device=device)
        l_prob, l_lbl = [], []

        with torch.no_grad():
            for batch in val_loader:
                if isinstance(batch[1], dict):
                    images    = batch[0].to(device, non_blocking=True)
                    tb_labels = batch[1]["labels_a"].to(device, non_blocking=True)
                    findings  = torch.zeros(images.shape[0], 6, device=device)
                    active    = torch.full((images.shape[0],), -1, dtype=torch.long, device=device)
                else:
                    images, tb_raw, find_raw, act_raw, *_ = batch
                    images    = images.to(device, non_blocking=True)
                    tb_labels = tb_raw.to(device, non_blocking=True)
                    findings  = find_raw.to(device, non_blocking=True)
                    active    = act_raw.to(device, non_blocking=True)
                out = model(images)
                vl_t += criterion(out, tb_labels, findings, active)["loss"].detach()
                l_prob.extend(out["tb_prob"].cpu().tolist())
                l_lbl.extend((tb_labels.argmax(-1) if tb_labels.dim() > 1 else tb_labels).cpu().tolist())

        dist.all_reduce(vl_t, op=dist.ReduceOp.AVG)
        avg_val = float(vl_t / max(1, len(val_loader)))

        # gather all-rank predictions for global AUC
        all_p = [None] * world_size
        all_l = [None] * world_size
        dist.all_gather_object(all_p, l_prob)
        dist.all_gather_object(all_l, l_lbl)

        # rank 0: compute metrics + save
        if is_main:
            flat_p = [v for s in all_p for v in s]
            flat_l = [v for s in all_l for v in s]
            val_auc = 0.5
            if _roc is not None and len(set(flat_l)) > 1:
                try:
                    val_auc = float(_roc(np.array(flat_l), np.array(flat_p)))
                except Exception:
                    pass

            row = dict(epoch=epoch, train_loss=avg_train, val_loss=avg_val,
                       val_auc=val_auc, elapsed_s=time.time() - t0)
            history.append(row)
            print(f"Epoch {epoch:03d} | train={avg_train:.4f} | val_loss={avg_val:.4f} | val_auc={val_auc:.4f} | {row['elapsed_s']:.0f}s")

            if val_auc > best_auc:
                best_auc = val_auc; no_improve = 0
                try:
                    sd = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
                    torch.save({"epoch": epoch, "model_state_dict": sd,
                                "optimizer_state_dict": optimizer.state_dict(),
                                "val_auc": val_auc, "history": history},
                               out_dir / "best_model.pt")
                except Exception as e:
                    warnings.warn(f"Checkpoint save failed: {e}")
            else:
                no_improve += 1

            try:
                sd = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
                torch.save({"epoch": epoch, "model_state_dict": sd},
                           out_dir / "last_model.pt")
            except Exception:
                pass
            should_stop = int(no_improve >= pat)
        else:
            should_stop = 0

        stop_t = torch.tensor(should_stop, dtype=torch.int32, device=device)
        dist.broadcast(stop_t, src=0)
        dist.barrier()

        if stop_t.item():
            if is_main:
                print(f"Early stopping at epoch {epoch}")
            break

    if is_main:
        print(f"\nDone. Best val AUC = {best_auc:.4f}")
        (out_dir / "history.json").write_text(json.dumps(history, indent=2))

    cleanup_ddp()


print("ddp_worker defined")


In [ ]:
# ============================================================
# 11. LAUNCH TRAINING  (single GPU)
#
# DDP with fork() is unreliable in Jupyter notebooks because
# the forked children can inherit a corrupted CUDA/NCCL context.
# Single-GPU training is simpler, fully utilises the T4 16GB
# VRAM, and reaches effective batch=64 via gradient accumulation.
#
# To re-enable DDP: set USE_DDP = True (requires a clean kernel
# with no prior CUDA calls before this cell).
# ============================================================
USE_DDP = False   # set True only if DDP is confirmed working

wandb_run = None
try:
    import wandb
    key = os.environ.get("WANDB_API_KEY", "") or ""
    if not key:
        try:
            from kaggle_secrets import UserSecretsClient
            key = UserSecretsClient().get_secret("WANDB_API_KEY") or ""
        except ImportError:
            pass
    if key:
        wandb.login(key=key, relogin=True)
        wandb_run = wandb.init(
            project="cough-vision",
            name=f'{HP["preset"]}_{HP["max_epochs"]}ep_1gpu',
            config=HP, dir=str(WORK_DIR),
        )
        print("W&B:", wandb_run.url)
    else:
        print("W&B skipped (add WANDB_API_KEY to Kaggle Secrets or env var)")
except ImportError:
    print("wandb not installed")

CKPT_DIR = str(WORK_DIR / "checkpoints")

ARGS = dict(
    src_root        = str(SRC_ROOT),
    preset          = HP["preset"],
    cnn_backbone    = HP["cnn_backbone"],
    vit_backbone    = HP["vit_backbone"],
    pretrained      = HP["pretrained"],
    max_epochs      = HP["max_epochs"],
    freeze_epochs   = HP["freeze_epochs"],
    batch_size      = HP["batch_size"],
    accum_steps     = HP["accum_steps"],
    backbone_lr     = HP["backbone_lr"],
    head_lr         = HP["head_lr"],
    weight_decay    = HP["weight_decay"],
    warmup_epochs   = HP["warmup_epochs"],
    patience        = HP["patience"],
    focal_gamma     = HP["focal_gamma"],
    focal_alpha     = HP["focal_alpha"],
    label_smoothing = HP["label_smoothing"],
    cls_weight      = HP["cls_weight"],
    findings_weight = HP["findings_weight"],
    active_weight   = HP["active_weight"],
    use_cutmix      = HP["use_cutmix"],
    use_mixup       = HP["use_mixup"],
    pos_weight      = HP["pos_weight"],
    mixed_precision = HP["mixed_precision"],
    seed            = HP["seed"],
    num_workers     = HP["num_workers"],
    train_csv       = str(DATA_DIR / "train.csv"),
    val_csv         = str(DATA_DIR / "val.csv"),
    output_dir      = CKPT_DIR,
    port            = find_free_port(HP["port"]),
)

eff_bs = HP["batch_size"] * HP["accum_steps"]
print(f"Effective batch = {eff_bs}  ({HP['batch_size']} x {HP['accum_steps']} accum)")
print(f"Device          : cuda:0  ({torch.cuda.get_device_properties(0).name})")
print(f"DDP             : {'enabled (fork)' if USE_DDP and WORLD_SIZE > 1 else 'disabled (single GPU)'}")
print()

if USE_DDP and WORLD_SIZE > 1:
    # Fork-based DDP (Linux / Kaggle only).
    # Only enable if you have verified no CUDA is initialised before this cell.
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    ctx   = torch.multiprocessing.get_context("fork")
    procs = []
    for rank in range(WORLD_SIZE):
        p = ctx.Process(target=ddp_worker, args=(rank, WORLD_SIZE, ARGS))
        p.daemon = False
        p.start()
        procs.append(p)
        print(f"  Launched rank={rank} pid={p.pid}")

    failed = False
    for rank, p in enumerate(procs):
        p.join()
        status = "OK" if p.exitcode == 0 else f"FAILED (exit {p.exitcode})"
        print(f"  Worker rank={rank}: {status}")
        if p.exitcode != 0:
            failed = True

    if failed:
        raise RuntimeError("One or more DDP workers failed.")
else:
    # Single GPU — default, reliable, uses full T4 16GB VRAM
    ddp_worker(rank=0, world_size=1, args=ARGS)

if wandb_run is not None:
    try: wandb_run.finish()
    except Exception: pass

print("\nTraining complete.")


In [ ]:
# ============================================================
# 12. TRAINING CURVES
# ============================================================
import matplotlib.pyplot as plt, pathlib

hist_p = pathlib.Path(CKPT_DIR) / "history.json"
if not hist_p.exists():
    bp = pathlib.Path(CKPT_DIR) / "best_model.pt"
    history = torch.load(str(bp), map_location="cpu").get("history", []) if bp.exists() else []
else:
    history = json.loads(hist_p.read_text())

if not history:
    print("No history -- run training first.")
else:
    ep  = [h["epoch"] for h in history]
    tl  = [h["train_loss"] for h in history]
    vl  = [h["val_loss"]   for h in history]
    auc = [h["val_auc"]    for h in history]
    best_ep = max(history, key=lambda h: h["val_auc"])["epoch"]
    best_a  = max(h["val_auc"] for h in history)

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    a1.plot(ep, tl, label="Train loss", color="steelblue", lw=2)
    a1.plot(ep, vl, label="Val loss",   color="tomato",    lw=2)
    a1.axvline(HP["freeze_epochs"], color="gray", ls="--", alpha=0.6)
    a1.set(xlabel="Epoch", ylabel="Loss", title="Loss"); a1.legend(); a1.grid(alpha=0.3)

    a2.plot(ep, auc, color="forestgreen", marker="o", ms=3, lw=2)
    a2.axhline(0.90, color="red", ls="--", lw=1.5, label="WHO 90%")
    a2.scatter([best_ep], [best_a], color="gold", s=120, zorder=5,
               label=f"Best={best_a:.4f} ep{best_ep}")
    a2.set(xlabel="Epoch", ylabel="AUC-ROC", title="AUC (all GPUs)"); a2.legend(); a2.grid(alpha=0.3)

    plt.suptitle("cough-vision Training", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(WORK_DIR / "training_curves.png", dpi=120)
    plt.show()
    print(f"Best AUC = {best_a:.4f} @ epoch {best_ep}")


In [ ]:
# ============================================================
# 13. EVALUATION -- WHO TPP REPORT
# ============================================================
import numpy as np
from evaluation.metrics import evaluate, find_who_threshold
from models import build_ensemble
from data.dataset import TBDataset
from data.augmentation import get_inference_transform
from torch.utils.data import DataLoader

ckpt_p = pathlib.Path(CKPT_DIR) / "best_model.pt"
_ecfg  = get_config(HP["preset"])
_ecfg.cnn.name = HP["cnn_backbone"]
_ecfg.vit.name = HP["vit_backbone"]
eval_model = build_ensemble(_ecfg)

if ckpt_p.exists():
    ckpt = torch.load(str(ckpt_p), map_location="cpu")
    eval_model.load_state_dict(ckpt["model_state_dict"])
    print(f"Loaded epoch={ckpt.get('epoch','?')} val_auc={ckpt.get('val_auc',0):.4f}")
else:
    print("WARNING: no checkpoint")

eval_dev   = torch.device("cuda:0" if WORLD_SIZE > 0 else "cpu")
eval_model = eval_model.to(eval_dev).eval()

test_ds = TBDataset(csv_path=DATA_DIR/"test.csv", image_root=pathlib.Path("/"),
                    split="test", transform=get_inference_transform(224))
test_loader = DataLoader(test_ds, batch_size=HP["batch_size"]*2,
                         shuffle=False, num_workers=HP["num_workers"],
                         pin_memory=(eval_dev.type=="cuda"))

all_probs, all_lbls = [], []
with torch.no_grad():
    for batch in test_loader:
        imgs, lbls, *_ = batch
        out = eval_model(imgs.to(eval_dev))
        all_probs.extend(out["tb_prob"].cpu().tolist())
        all_lbls.extend(lbls.cpu().tolist())

y_true = np.array(all_lbls,  dtype=int)
y_prob = np.array(all_probs, dtype=float)

report = evaluate(y_true, y_prob, sensitivity_target=HP["who_sensitivity"],
                  n_bootstrap=1000, site="test_set")
(WORK_DIR / "eval_report.json").write_text(json.dumps(report, indent=2, default=str))

print("=" * 50)
try:
    print(f"AUC-ROC : {report['auc']['auc_roc']:.4f}  "
          f"CI [{report['auc_roc_ci']['ci_lower']:.4f}, {report['auc_roc_ci']['ci_upper']:.4f}]")
    print(f"AUC-PR  : {report['auc']['auc_pr']:.4f}")
except (KeyError, TypeError): pass
try:
    who  = report["who_operating_point"]
    mets = report["metrics_at_who_threshold"]
    print(f"WHO TPP : {'MET' if who.get('who_tpp_met') else 'NOT MET'}")
    print(f"  threshold={who['threshold']:.4f}  sens={who['sensitivity']:.1%}  spec={who['specificity']:.1%}")
    print(f"  PPV={mets.get('ppv',0):.1%}  NPV={mets.get('npv',0):.1%}  F1={mets.get('f1',0):.4f}")
except (KeyError, TypeError): pass
try:
    cal = report["calibration"]
    print(f"ECE={cal['ece']:.4f}  Brier={cal['brier']:.4f}")
except (KeyError, TypeError): pass
print("=" * 50)


In [ ]:
# ============================================================
# 14. ROC + PR CURVES
# ============================================================
from sklearn.metrics import roc_curve, precision_recall_curve, auc

fpr, tpr, _ = roc_curve(y_true, y_prob)
prec, rec, _ = precision_recall_curve(y_true, y_prob)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 6))
a1.plot(fpr, tpr, lw=2, color="steelblue", label=f"AUC={auc(fpr,tpr):.4f}")
a1.plot([0,1],[0,1],"k--",alpha=0.4); a1.axhline(0.90, color="red", ls=":", alpha=0.6)
a1.set(xlabel="FPR", ylabel="TPR", title="ROC"); a1.legend(); a1.grid(alpha=0.3)

a2.plot(rec, prec, lw=2, color="forestgreen", label=f"PR AUC={auc(rec,prec):.4f}")
a2.axhline(y_true.mean(), color="gray", ls="--", label=f"Prev={y_true.mean():.1%}")
a2.set(xlabel="Recall", ylabel="Precision", title="PR Curve"); a2.legend(); a2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(WORK_DIR / "roc_pr_curves.png", dpi=120)
plt.show()


In [ ]:
# ============================================================
# 15. GRAD-CAM
# ============================================================
import cv2, PIL.Image as PIL_img
from data.preprocessing import load_image, apply_clahe
from data.augmentation import get_inference_transform

eval_model.enable_gradcam()
_tf = get_inference_transform(224)

cases = [r for r in test_ds.records if int(r.get("tb_label",0))==1][:4] or test_ds.records[:4]
fig, axes = plt.subplots(len(cases), 3, figsize=(13, 4*len(cases)))
if len(cases) == 1: axes = [axes]

for i, rec in enumerate(cases):
    raw  = load_image(rec["image_path"])
    H, W = raw.shape
    pil  = PIL_img.fromarray(apply_clahe(raw)).convert("RGB")
    img_t = _tf(pil).unsqueeze(0).to(eval_dev)

    eval_model.eval()
    with torch.no_grad():
        score = float(eval_model(img_t)["tb_score"].cpu())

    try:
        hmap    = eval_model._gradcam(img_t, class_idx=1)
        overlay = eval_model._gradcam.overlay(hmap, raw)
    except Exception as e:
        print(f"CAM error: {e}")
        hmap    = np.zeros((H, W))
        overlay = cv2.cvtColor(raw, cv2.COLOR_GRAY2RGB)

    axes[i][0].imshow(raw, cmap="gray"); axes[i][0].set_title(f"label={rec['tb_label']}"); axes[i][0].axis("off")
    im = axes[i][1].imshow(cv2.resize(hmap,(W,H)), cmap="jet", vmin=0, vmax=1)
    axes[i][1].set_title(f"Grad-CAM score={score:.1f}"); axes[i][1].axis("off")
    plt.colorbar(im, ax=axes[i][1], fraction=0.046)
    axes[i][2].imshow(overlay); axes[i][2].set_title("Overlay"); axes[i][2].axis("off")

plt.tight_layout()
plt.savefig(WORK_DIR / "gradcam_tb_cases.png", dpi=120)
plt.show()
eval_model.disable_gradcam()
print("Saved -> gradcam_tb_cases.png")


In [ ]:
# ============================================================
# 16. PER-SITE CALIBRATION
# ============================================================
from evaluation.calibration import build_calibrator
from evaluation.metrics import find_who_threshold

cal = build_calibrator("platt")
cal.fit(y_prob, y_true, sensitivity_target=HP["who_sensitivity"])

who_cal = find_who_threshold(y_true, cal.predict_proba(y_prob), HP["who_sensitivity"])
print(f"Threshold : {cal.threshold_:.4f}")
print(f"Sens      : {who_cal['sensitivity']:.1%}")
print(f"Spec      : {who_cal['specificity']:.1%}")
print(f"WHO TPP   : {'MET' if who_cal['who_tpp_met'] else 'NOT MET'}")

cal.save(WORK_DIR / "calibrator.json")
print("Saved -> calibrator.json")


In [ ]:
# ============================================================
# 17. ONNX EXPORT
# ============================================================
from deployment.export import export_onnx, optimise_onnx

try:
    exp = export_onnx(model=eval_model, output_path=WORK_DIR/"cough_vision.onnx",
                      cnn_input_size=224, opset=17, dynamic_axes=True, model_version="v1.0.0")
    print(f"ONNX exported ({exp.stat().st_size/1e6:.1f} MB) -> {exp}")
    try:
        optimise_onnx(exp)
    except ImportError:
        print("onnxoptimizer not installed -- skip optimisation")
except Exception as e:
    print(f"ONNX export failed: {e}")


In [ ]:
# ============================================================
# 18. OUTPUT ARTEFACTS
# ============================================================
for name, path in [
    ("best_model.pt",        pathlib.Path(CKPT_DIR) / "best_model.pt"),
    ("calibrator.json",      WORK_DIR / "calibrator.json"),
    ("eval_report.json",     WORK_DIR / "eval_report.json"),
    ("hyperparameters.json", WORK_DIR / "hyperparameters.json"),
    ("training_curves.png",  WORK_DIR / "training_curves.png"),
    ("roc_pr_curves.png",    WORK_DIR / "roc_pr_curves.png"),
    ("gradcam_tb_cases.png", WORK_DIR / "gradcam_tb_cases.png"),
    ("cough_vision.onnx",    WORK_DIR / "cough_vision.onnx"),
]:
    if path.exists():
        print(f"  SAVED  {name:<28} {path.stat().st_size/1e3:>8.1f} KB")
    else:
        print(f"  MISS   {name}")
print(f"\nAll output in: {WORK_DIR}")
